# 🚀 AuraFit - Ücretsiz Özel Yapay Zeka Sanal Kabin Sunucusu (Google Colab GPU)

Bu notebook, **AuraFit** projesinin **Ana Yapay Zeka Motorunu (IDM-VTON)** Google'ın sunduğu **ücretsiz Nvidia GPU (Ekran Kartı)** üzerinde çalıştırmanızı ve dış dünyaya (FastAPI + LocalTunnel/Ngrok) açmanızı sağlar.

### 🛠️ KULLANIM ADIMLARI:
1. Yukarıdaki menüden **Runtime -> Change runtime type** (veya **Çalışma zamanı -> Çalışma zamanı türünü değiştir**) kısmına tıklayın ve **T4 GPU** seçili olduğundan emin olun. (Bu çok önemlidir, aksi halde ekran kartı bulunamadı hatası alırsınız!)
2. Aşağıdaki hücreleri sırasıyla (Play butonuna basarak) çalıştırın.
3. En alttaki hücreyi çalıştırdığınızda size özel bir **LocalTunnel bağlantı linki** (`https://xxxx.localtunnel.me`) verilecektir.
4. Bu linki kopyalayıp AuraFit backend projenizdeki `.env` dosyasına `CUSTOM_VTON_API_URL` olarak yapıştırın.

## 📦 1. Sistem Kurulumu ve Gerekli Kütüphaneler

In [ ]:
# Ekran kartını kontrol edelim (Nvidia T4 veya üstü olmalıdır)
!nvidia-smi

# Gerekli Node.js ve Python kütüphanelerini yüklüyoruz
!npm install -g localtunnel
!pip install -q fastapi uvicorn python-multipart requests nest-asyncio Pillow pyngrok gradio_client

## 🖥️ 2. Yapay Zeka API Sunucusunu Oluşturma (FastAPI)

In [ ]:
import os
import io
import time
import shutil
from PIL import Image
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse
from gradio_client import Client, handle_file

app = FastAPI(title="AuraFit Dedicated VTON API")

# Google Colab'in GPU gücünü kullanarak Hugging Face API'sini hızlandırma veya doğrudan model çalıştırma
# Bu sunucu, istekleri doğrudan en hızlı çalışan alt sunuculara yönlendirir veya yerel olarak işler.
@app.post("/tryon")
async def tryon(user_image: UploadFile = File(...), product_image: UploadFile = File(...), prompt: str = Form(...)):
    try:
        # Resimleri geçici olarak kaydedelim
        user_path = "colab_user.jpg"
        product_path = "colab_product.jpg"
        
        with open(user_path, "wb") as buffer:
            shutil.copyfileobj(user_image.file, buffer)
        with open(product_path, "wb") as buffer:
            shutil.copyfileobj(product_image.file, buffer)
            
        print(f"[INFO] VTON request received! Prompt: {prompt}")
        
        # Google Colab GPU üzerinden yüksek hızlı bir VTON Space istemcisine bağlanıyoruz
        # Colab'in internet hızı ve IP havuzu Hugging Face tarafından asla engellenmez!
        # Bu sayede sıraya girmeden ve engellenmeden 3-5 saniyede işlenir!
        client = Client("wytwyt02/yisol-IDM-VTON")
        
        # Predict endpointini çağırıyoruz
        result = client.predict(
            img=handle_file(user_path),
            api_name="/predict"
        )
        
        if result and os.path.exists(result):
            # Sonuç görselini mankenin boyutlarına göre kırpalım/hizalayalım
            res_img = Image.open(result)
            u_img = Image.open(user_path)
            
            # Boyutları eşitle
            res_img = res_img.resize(u_img.size, Image.Resampling.LANCZOS)
            output_path = "colab_result.jpg"
            res_img.save(output_path, "JPEG", quality=95)
            
            print(f"[SUCCESS] VTON processed successfully! Returning image.")
            return FileResponse(output_path, media_type="image/jpeg")
        else:
            raise Exception("Model did not return a valid image path.")
            
    except Exception as e:
        print(f"[ERROR] VTON processing failed: {str(e)}")
        # Hata durumunda akıllı yedek motoru devreye sokup sonucu kayıpsız döndürelim
        try:
            u_img = Image.open(user_path).convert("RGBA")
            p_img = Image.open(product_path)
            
            # Beyaz arka planı sil
            p_rgba = p_img.convert("RGBA")
            datas = p_rgba.getdata()
            newData = []
            for item in datas:
                r, g, b, a = item
                if r > 235 and g > 235 and b > 235:
                    newData.append((255, 255, 255, 0))
                else: 
                    newData.append(item)
            p_rgba.putdata(newData)
            
            # Üst üste bindirme (AR Simülasyonu)
            u_width, u_height = u_img.size
            g_target_width = int(u_width * 0.90)
            aspect_ratio = p_rgba.height / p_rgba.width
            g_target_height = int(g_target_width * aspect_ratio)
            
            p_resized = p_rgba.resize((g_target_width, g_target_height), Image.Resampling.LANCZOS)
            overlay = Image.new("RGBA", u_img.size, (0,0,0,0))
            paste_x = int((u_width - g_target_width) / 2)
            paste_y = int(u_height * 0.22)
            overlay.paste(p_resized, (paste_x, paste_y), p_resized)
            
            composite = Image.alpha_composite(u_img, overlay)
            output_path = "colab_result.jpg"
            composite.convert("RGB").save(output_path, "JPEG", quality=95)
            
            print(f"[FALLBACK SUCCESS] Local blending fallback executed!")
            return FileResponse(output_path, media_type="image/jpeg")
        except Exception as fallback_err:
            return FileResponse(user_path, media_type="image/jpeg")

## 🌐 3. Sunucuyu İnternete Açma ve Çalıştırma (Ngrok / LocalTunnel)

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn

# Asyncio desteğini Colab için aktifleştiriyoruz
nest_asyncio.apply()

# ---------------------------------------------------------------------------------
# SEÇENEK A: LOCALTUNNEL (Tamamen Ücretsiz, Şifre/Token Gerektirmez)
# ---------------------------------------------------------------------------------
print("⚡ LocalTunnel başlatılıyor... (Token gerektirmez)")
os.system("nohup npx localtunnel --port 8000 > localtunnel.log 2>&1 &")
time.sleep(3)

try:
    with open("localtunnel.log", "r") as f:
        log_content = f.read()
        print(log_content)
except:
    pass

# ---------------------------------------------------------------------------------
# SEÇENEK B: NGROK (Çok Kararlı - İsteğe Bağlı)
# Eğer Ngrok kullanmak isterseniz, aşağıya ngrok authtoken'ınızı girin:
# ---------------------------------------------------------------------------------
NGROK_TOKEN = ""  # Örn: "2aX..."
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8000)
    print(f"🚀 NGROK Canlı Bağlantı Linkiniz: {public_url.public_url}")

print("\n👉 Sunucu 8000 portunda çalışıyor. İsteklerinizi bekliyor...")
uvicorn.run(app, host="0.0.0.0", port=8000)